# Fine-Tuning Prometheus (LLM-as-a-Judge) con LoRA

Este notebook muestra cómo realizar fine-tuning al modelo Prometheus usando LoRA (Low-Rank Adaptation).

In [ ]:
import sys
import os
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# Añadir src al path
sys.path.append(os.path.abspath(os.path.join('../src')))

from utils import load_data
from model_utils import get_model_and_tokenizer

In [ ]:
# Configuración
MODEL_NAME = "prometheus-eval/prometheus-7b-v2.0" # O el modelo base que prefieras
OUTPUT_DIR = "../output/prometheus_finetuned"
DATA_PATH = "../data/dataset.json"

In [ ]:
# Cargar datos
df = load_data(DATA_PATH)
dataset = Dataset.from_pandas(df)
dataset

In [ ]:
# Formatear datos para Prometheus
# Nota: Ajusta este formato según la versión específica de Prometheus y tus necesidades.
def format_instruction(sample):
    # Ejemplo de formato de prompt para Prometheus
    prompt = f"###Task Description:\nAnswering a question.\n\n###The Instruction:\n{sample['question']}\n\n###The Response:\n{sample['proposal_answer']}\n\n###The Reference Answer:\n{sample['answer']}\n\n###Score:\n"
    
    # Aquí asumimos que 'verdict' contiene el feedback y score si aplica.
    # Si 'verdict' es solo texto, el modelo aprenderá a generar ese texto.
    output = f"{sample['verdict']}"
    
    return {
        "text": prompt + output,
        "input_text": prompt,
        "output_text": output
    }

dataset = dataset.map(format_instruction)

In [ ]:
# Cargar Modelo y Tokenizer
# Nota: Asegúrate de configurar HF_TOKEN en .env
model, tokenizer = get_model_and_tokenizer(MODEL_NAME, load_in_4bit=True)

# Si el modelo es None (por la protección en script), esto fallará hasta que descomentes la carga real en src/model_utils.py
if model is None:
    print("DETENIDO: Configura src/model_utils.py para cargar el modelo real.")
else:
    tokenizer.padding_side = 'right' # O 'left' dependiendo del modelo, usualmente right para generacion
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Configuración LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"] # Ajustar según arquitectura del modelo (e.g. Llama usa q_proj, v_proj)
)

if model is not None:
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

In [ ]:
# Tokenización
def tokenize_function(examples):
    # Concatenar input y output para CausalLM es común, o usar DataCollatorForCompletionOnlyLM
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=512)

if model is not None:
    tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
# Entrenamiento
if model is not None:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        num_train_epochs=1,
        save_strategy="epoch",
        fp16=True,
        optim="paged_adamw_8bit"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets,
        data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt"),
    )

    # trainer.train() # Descomentar para entrenar
    print("Entrenamiento configurado. Descomenta trainer.train() para iniciar.")